# 04 — Simulation & Stability (NOT real findings)

⚠️ **Read this first.** The real survey sample (n≈55) is too small for reliable predictive modeling
(see `03_models.ipynb`). This notebook does two things, honestly labeled:

1. **Bootstrap stability check** — resamples the *real* data thousands of times to see which
   patterns are stable and which could be noise. This is legitimate statistics on real data.
2. **Synthetic scale-up demo** — generates a 10x synthetic dataset from the real distributions
   to demonstrate how the ML pipeline behaves with more data. **Synthetic rows are not students.
   Any pattern the models find here was inherited from the real 55 by construction — this
   demonstrates the pipeline, it does not confirm the findings.**

All findings and advice in this project come from the real sample only (notebooks 01–03).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor

data = pd.read_csv('../data/clean.csv')
FEATURES = ['study_hours_n', 'sleep_n', 'consistency', 'ai_freq', 'ai_confidence',
            'ai_dependence', 'ai_verify', 'difficulty', 'distraction', 'independence']
data = data[FEATURES + ['gpa']].dropna().reset_index(drop=True)
rng = np.random.default_rng(42)
print(f"Real sample: n = {len(data)}")

## Part 1 — Bootstrap: which patterns are stable?

Resample the real 55 with replacement 2,000 times; recompute each correlation with GPA every time.
The 95% interval shows how much each estimate could move with a different sample of students.

In [ ]:
KEY = ['study_hours_n', 'sleep_n', 'ai_verify', 'ai_freq', 'ai_dependence']
boots = {f: [] for f in KEY}
for _ in range(2000):
    s = data.sample(len(data), replace=True)
    for f in KEY:
        boots[f].append(s[f].corr(s['gpa']))

for f in KEY:
    lo, hi = np.percentile(boots[f], [2.5, 97.5])
    mid = np.mean(boots[f])
    print(f"{f:15s} corr = {mid:+.2f}   95% CI [{lo:+.2f}, {hi:+.2f}]")

**How to read this (results on the current sample):**
- `study_hours` is the most stable positive pattern — its interval barely touches zero.
- `sleep` and `ai_verify` lean positive but their intervals cross zero → suggestive, not confirmed.
- `ai_freq` is centered on ~zero → the "usage frequency doesn't matter" null result is itself stable.

This is exactly why the project reports *patterns, not proof*.

## Part 2 — Synthetic scale-up: what would the pipeline do with 10x the data?

We generate 550 synthetic rows by resampling real rows and adding noise, then rerun the
random forest. **Demo only** — the synthetic data inherits the real sample's patterns.

In [ ]:
synth = data.sample(len(data) * 10, replace=True).reset_index(drop=True)
for c in FEATURES:
    synth[c] = synth[c] + rng.normal(0, data[c].std() * 0.15, len(synth))
synth['gpa'] = (synth['gpa'] + rng.normal(0, 0.08, len(synth))).clip(0, 4)

cv = KFold(5, shuffle=True, random_state=42)
rf = RandomForestRegressor(n_estimators=300, max_depth=4, random_state=42)
r2_real = cross_val_score(rf, data[FEATURES], data['gpa'], cv=cv, scoring='r2').mean()
r2_syn = cross_val_score(rf, synth[FEATURES], synth['gpa'], cv=cv, scoring='r2').mean()
print(f"Random forest R2 — real data (n={len(data)}):        {r2_real:.3f}")
print(f"Random forest R2 — synthetic 10x (n={len(synth)}):  {r2_syn:.3f}")

In [ ]:
# Which features does the synthetic-trained model lean on?
rf.fit(synth[FEATURES], synth['gpa'])
pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False).round(3)

## Conclusion

- On real data, cross-validated R² is negative → the models cannot reliably predict an
  individual's GPA at this sample size. That negative result is reported honestly.
- With 10x data of the same shape, the pipeline trains stably (R² ≈ 0.5 on the demo) —
  the code is ready for a larger real sample whenever one exists.
- The project's practical advice therefore rests on the **descriptive findings from real
  students** (see `02_exploration.ipynb` and `figures/`), with bootstrap intervals above
  showing which of those patterns are most stable.